## Importing Packages

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier

pd.set_option('display.max_rows',100)

## Import Feature Engineered Dataset

In [2]:
df = pd.read_pickle('../data/lending_club_feature_engineered.pkl')

print(df.shape)
df.head()

(1340973, 128)


,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,emp_title,emp_length,...,repayment_capacity_score,creditworthiness_score,financial_stability_Score,loan_to_income_ratio_capped,funded_to_income_ratio_capped,revol_bal_to_income_ratio_capped,installment_to_income_ratio_capped,open_acc_to_total_acc_ratio_capped,debt_burden_score_capped,financial_stability_score
100,30000,30000,30000.0,36 months,22.35,1151.16,D,D5,Supervisor,5 years,...,-0.860379,3.011861,-0.846094,0.300000,0.300000,0.156030,0.138139,0.578947,67.46,-0.846094
152,40000,40000,40000.0,60 months,16.14,975.71,C,C4,Assistant to the Treasurer (Payroll),< 1 year,...,-1.759874,2.087647,NaN,0.500000,0.500000,0.777133,0.200983,0.486486,115.03,NaN
170,20000,20000,20000.0,36 months,7.56,622.68,A,A3,Teacher,10+ years,...,-0.220578,0.975468,0.706151,0.200000,0.200000,0.254160,0.074722,0.473684,48.82,0.706151
186,4500,4500,4500.0,36 months,11.31,147.99,B,B3,Accounts Examiner III,10+ years,...,-0.322414,1.441015,1.181911,0.116883,0.116883,0.116156,0.046127,0.480000,19.94,1.181911
215,8425,8425,8425.0,36 months,27.27,345.18,E,E5,Senior Director Risk Management,3 years,...,0.986400,2.714255,-0.018850,0.025688,0.025667,0.081804,0.010252,0.567568,78.07,-0.018850


## Target Variable

In [3]:
target = df['default_flag']

df['default_flag'].value_counts()


default_flag
0    1043940
1     297033
Name: count, dtype: int64

In [4]:
df['default_flag'].value_counts(normalize=True)

default_flag
0    0.778494
1    0.221506
Name: proportion, dtype: float64

## Create Stratified sampling for Faster Development

In [5]:
df_sample,_ = train_test_split(df, train_size=200000, stratify=df['default_flag'], random_state=42)
df=df_sample.copy()

print(f'Sample dataset shape {df.shape}')
print(df['default_flag'].value_counts())
print(df['default_flag'].value_counts(normalize=True))

Sample dataset shape (200000, 128)
default_flag
0    155699
1     44301
Name: count, dtype: int64
default_flag
0    0.778495
1    0.221505
Name: proportion, dtype: float64


## Remove Obvious Leakage Columns 

In [6]:
leakage_cols = [

    "out_prncp",
    "out_prncp_inv",

    "total_pymnt",
    "total_pymnt_inv",

    "total_rec_prncp",
    "total_rec_int",

    "recoveries",
    "collection_recovery_fee",

    "last_pymnt_amnt"
]

available_leakage_cols =  [cols for cols in leakage_cols if cols in df.columns]
df = df.drop(columns=available_leakage_cols)
print(df.shape)

(200000, 119)


## Dependent & Independent Variables

In [7]:
X = df.drop(columns='default_flag')
y = df['default_flag']

print(f'Shape of X {X.shape}')
print(f'Shape of y {y.shape}')

Shape of X (200000, 118)
Shape of y (200000,)


## Feature Types

In [9]:
cat_cols = X.select_dtypes(include="object").columns.to_list()
num_cols = X.select_dtypes(include=["int64","int32","float64"]).columns.to_list()

print(f'Categorical Column Counts = {len(cat_cols)}')
print(f'Numerical Column Counts = {len(num_cols)}')

for col in cat_cols:
    print(col, X[col].nunique())

Categorical Column Counts = 14
Numerical Column Counts = 100
term 2
grade 7
sub_grade 35
emp_title 79309
emp_length 11
home_ownership 6
verification_status 3
purpose 14
addr_state 51
initial_list_status 2
application_type 2
hardship_flag 2
disbursement_method 2
debt_settlement_flag 2


## Train Test Split

In [10]:
X_train, X_test, y_train, y_test = train_test_split(X,y,train_size=0.80,stratify=y,random_state=42)
print(X_train.shape)
print(X_test.shape)

(160000, 118)
(40000, 118)


## Handling Missing Values

### Numerical Variables

In [12]:
num_impute = SimpleImputer(strategy='median')
X_train_num = pd.DataFrame(num_impute.fit_transform(X_train[num_cols]), columns=num_cols, index=X_train.index)
X_test_num = pd.DataFrame(num_impute.transform(X_test[num_cols]), columns=num_cols, index=X_test.index)

### Categorical Variables

In [13]:
cat_impute = SimpleImputer(strategy='most_frequent')
X_train_cat = pd.DataFrame(cat_impute.fit_transform(X_train[cat_cols]),columns=cat_cols, index=X_train.index)
X_test_cat = pd.DataFrame(cat_impute.transform(X_test[cat_cols]), columns=cat_cols, index=X_test.index)

## Drop High Cardinality/ Leakage Categorical Columns

In [14]:
drop_cat_columns = ['emp_title','hardship_flag','debt_settlement_flag']

X_train_cat.drop(columns=drop_cat_columns,inplace=True)
X_test_cat.drop(columns=drop_cat_columns,inplace=True)

print('Remaining Categorical Columns:')
for col in X_train_cat.columns:
    print(col, X_train_cat[col].nunique())

Remaining Categorical Columns:
term 2
grade 7
sub_grade 35
emp_length 11
home_ownership 6
verification_status 3
purpose 14
addr_state 51
initial_list_status 2
application_type 2
disbursement_method 2


## One Hot Encoding

In [15]:
X_train_cat_encoded = pd.get_dummies(X_train_cat, drop_first=True)
X_test_cat_encoded = pd.get_dummies(X_test_cat, drop_first=True)

In [16]:
X_train_cat_encoded.columns.to_list()

['term_ 60 months',
 'grade_B',
 'grade_C',
 'grade_D',
 'grade_E',
 'grade_F',
 'grade_G',
 'sub_grade_A2',
 'sub_grade_A3',
 'sub_grade_A4',
 'sub_grade_A5',
 'sub_grade_B1',
 'sub_grade_B2',
 'sub_grade_B3',
 'sub_grade_B4',
 'sub_grade_B5',
 'sub_grade_C1',
 'sub_grade_C2',
 'sub_grade_C3',
 'sub_grade_C4',
 'sub_grade_C5',
 'sub_grade_D1',
 'sub_grade_D2',
 'sub_grade_D3',
 'sub_grade_D4',
 'sub_grade_D5',
 'sub_grade_E1',
 'sub_grade_E2',
 'sub_grade_E3',
 'sub_grade_E4',
 'sub_grade_E5',
 'sub_grade_F1',
 'sub_grade_F2',
 'sub_grade_F3',
 'sub_grade_F4',
 'sub_grade_F5',
 'sub_grade_G1',
 'sub_grade_G2',
 'sub_grade_G3',
 'sub_grade_G4',
 'sub_grade_G5',
 'emp_length_10+ years',
 'emp_length_2 years',
 'emp_length_3 years',
 'emp_length_4 years',
 'emp_length_5 years',
 'emp_length_6 years',
 'emp_length_7 years',
 'emp_length_8 years',
 'emp_length_9 years',
 'emp_length_< 1 year',
 'home_ownership_MORTGAGE',
 'home_ownership_NONE',
 'home_ownership_OTHER',
 'home_ownership_OWN

In [17]:
print(f'X_train_cat_encoded shape is {X_train_cat_encoded.shape}')
print(f'X_test_cat_encoded shape is {X_test_cat_encoded.shape}')

X_train_cat_encoded shape is (160000, 124)
X_test_cat_encoded shape is (40000, 123)


In [18]:
X_train_cat_encoded, X_test_cat_encoded= X_train_cat_encoded.align(X_test_cat_encoded, join='left', axis=1, fill_value=0)

In [19]:
print(f'X_train_cat_encoded shape is {X_train_cat_encoded.shape}')
print(f'X_test_cat_encoded shape is {X_test_cat_encoded.shape}')

X_train_cat_encoded shape is (160000, 124)
X_test_cat_encoded shape is (40000, 124)


## Combine Numerical & Categorical Columns 

In [20]:
X_train_processed = pd.concat([X_train_num, X_train_cat_encoded], axis=1)
X_test_processed = pd.concat([X_test_num, X_test_cat_encoded], axis=1)

print(f'X_train_processed: {X_train_processed.shape}')
print(f'X_test_processed: {X_test_processed.shape}')

X_train_processed: (160000, 224)
X_test_processed: (40000, 224)


## Final Missing Value Check

In [21]:
print(f'Missing Value in Train: {X_train_processed.isnull().sum().sum()}')
print(f'Missing Value in Train: {X_test_processed.isnull().sum().sum()}')

Missing Value in Train: 0
Missing Value in Train: 0


## Evaluation Function

In [27]:
def evaluate_model(model_name, y_true, y_pred, y_proba):
    result = {
        'model': model_name,
        'accuracy': accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall': recall_score(y_true, y_pred, zero_division=0),
        'f1_score': f1_score(y_true, y_pred, zero_division=0),
        'roc_auc': roc_auc_score(y_true, y_proba)
    }

    print(f'\n*********{model_name}*********')
    print('Confusion Matrix:')
    print(confusion_matrix(y_true, y_pred))

    print('\nClassification Report')
    print(classification_report(y_true, y_pred, zero_division=0))

    print("ROC-AUC:", result["roc_auc"])

    return result

## Baseline Model - Logistic Regression

In [28]:
log_reg = LogisticRegression(max_iter=300, class_weight='balanced', random_state=42, n_jobs=-1)

log_reg.fit(X_train_processed, y_train)

log_pred = log_reg.predict(X_test_processed)
log_pred_proba = log_reg.predict_proba(X_test_processed)[:,1]

log_results = evaluate_model("Logistic Regression", y_test, log_pred, log_pred_proba)
print(log_results)

/Users/shankariseethalaksmimohanakrishnan/miniforge3/envs/ml_env/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 300 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=300).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(



*********Logistic Regression*********
Confusion Matrix:
[[19269 11871]
 [ 3783  5077]]

Classification Report
              precision    recall  f1-score   support

           0       0.84      0.62      0.71     31140
           1       0.30      0.57      0.39      8860

    accuracy                           0.61     40000
   macro avg       0.57      0.60      0.55     40000
weighted avg       0.72      0.61      0.64     40000

ROC-AUC: 0.6358500386371314
{'model': 'Logistic Regression', 'accuracy': 0.60865, 'precision': 0.299563370309181, 'recall': 0.5730248306997743, 'f1_score': 0.393443893366398, 'roc_auc': 0.6358500386371314}


## Ensemble Bagging Technique - Random Forest

In [30]:
rf_model = RandomForestClassifier(n_estimators=100, max_depth=8, class_weight='balanced', random_state=42, n_jobs=-1)

rf_model.fit(X_train_processed, y_train)

rf_pred = rf_model.predict(X_test_processed)
rf_pred_proba = rf_model.predict_proba(X_test_processed)[:,1]

rf_results = evaluate_model("Random Forest Classifier", y_test, rf_pred, rf_pred_proba)
print(rf_results)


*********Random Forest Classifier*********
Confusion Matrix:
[[20184 10956]
 [ 2772  6088]]

Classification Report
              precision    recall  f1-score   support

           0       0.88      0.65      0.75     31140
           1       0.36      0.69      0.47      8860

    accuracy                           0.66     40000
   macro avg       0.62      0.67      0.61     40000
weighted avg       0.76      0.66      0.69     40000

ROC-AUC: 0.7329930982340004
{'model': 'Random Forest Classifier', 'accuracy': 0.6568, 'precision': 0.35719314714855666, 'recall': 0.6871331828442437, 'f1_score': 0.47004323656578134, 'roc_auc': 0.7329930982340004}


## Ensemble Boosting Technique - LightGBM Model

In [32]:
lgbm_model = LGBMClassifier(learning_rate=0.05, n_estimators=200, max_depth=8, max_bin=255, class_weight='balanced',
                           random_state=42, n_jobs=-1, verbose=-1)

lgbm_model.fit(X_train_processed, y_train)

lgbm_pred = lgbm_model.predict(X_test_processed)
lgbm_pred_proba = lgbm_model.predict_proba(X_test_processed)[:,1]

lgbm_results = evaluate_model("LightGBM", y_test, lgbm_pred, lgbm_pred_proba)
print(lgbm_results)


*********LightGBM*********
Confusion Matrix:
[[21868  9272]
 [ 2886  5974]]

Classification Report
              precision    recall  f1-score   support

           0       0.88      0.70      0.78     31140
           1       0.39      0.67      0.50      8860

    accuracy                           0.70     40000
   macro avg       0.64      0.69      0.64     40000
weighted avg       0.77      0.70      0.72     40000

ROC-AUC: 0.7624621711313213
{'model': 'LightGBM', 'accuracy': 0.69605, 'precision': 0.39184048274957367, 'recall': 0.6742663656884876, 'f1_score': 0.4956442379490583, 'roc_auc': 0.7624621711313213}


## Model Comparision

In [37]:
model_results = [log_results, rf_results, lgbm_results]

results_df = pd.DataFrame(model_results)

results_df = results_df.sort_values(by='roc_auc', ascending=False)

results_df

,model,accuracy,precision,recall,f1_score,roc_auc
2,LightGBM,0.69605,0.391840,0.674266,0.495644,0.762462
1,Random Forest Classifier,0.65680,0.357193,0.687133,0.470043,0.732993
0,Logistic Regression,0.60865,0.299563,0.573025,0.393444,0.635850


## Save Processed Data & Model Results

In [38]:
X_train_processed.to_pickle('../data/X_train_processed.pkl')
X_test_processed.to_pickle('../data/X_test_processed.pkl')